## Settings

In [1]:
import sys
import os

# ensure local baseline package is importable
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from baseline.data import load_sequences, load_labels
from baseline.template_model import TemplateRepository
from baseline.search import seq_identity
from baseline.predict import generate_submission

sys.path.append(os.path.join(os.getcwd(), '..', 'scripts'))
from kabsch_utils import align_and_rmsd
from backbone_utils import extract_C1p_coords, extract_coords_from_submission

## Load Data

In [4]:
data_dir = os.path.join('..', 'data', 'stanford-rna-3d-folding-2')
print('data_dir ->', data_dir)

valid_seq = load_sequences(os.path.join(data_dir, 'validation_sequences.csv'))
train_seq = load_sequences(os.path.join(data_dir, 'train_sequences.csv'))
train_labels = load_labels(os.path.join(data_dir, 'train_labels.csv'))
print('loaded:', valid_seq.shape, train_seq.shape, train_labels.shape)

data_dir -> ../data/stanford-rna-3d-folding-2
loaded: (28, 2) (5716, 2) (7794971, 9)


## Create template repository

In [3]:
# Fit repository and generate a submission
repo = TemplateRepository()

repo.fit(train_seq, train_labels)
print('templates count:', len(repo.templates))

templates count: 5716


## Predict on validation set

In [ ]:
# NOTE: 並列化の意味は小さいかも
valid_pred = generate_submission(valid_seq, repo, seq_identity, n_structures=5, n_jobs=4)
print('submission rows:', len(valid_pred))

valid_pred.head()

,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,12.133,-11.862,1.757,166.708,162.080,161.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8ZNQ_2,C,2,NaN,NaN,NaN,167.160,157.024,159.502,193.176,143.370,155.162,NaN,NaN,NaN,9.981,-13.553,-4.518
2,8ZNQ_3,C,3,NaN,NaN,NaN,164.384,147.392,163.683,188.045,141.202,155.911,NaN,NaN,NaN,8.575,-8.310,5.003
3,8ZNQ_4,G,4,11.260,-7.167,5.547,160.163,145.020,166.617,NaN,NaN,NaN,-15.397,-2.768,4.068,12.275,-5.392,7.117
4,8ZNQ_5,U,5,12.239,-0.828,5.049,155.045,145.136,168.216,183.603,138.622,153.965,-6.594,-8.409,3.503,17.668,-2.604,8.109


## Evaluate on validation set

In [6]:
# Evaluation helper: imported from external module
import os
import pandas as pd
import numpy as np
from evaluate import evaluate_submission

In [7]:
valid_labels = load_labels(os.path.join(data_dir, 'validation_labels.csv'))

In [13]:
df, summary = evaluate_submission(pred_df=valid_pred, true_df=valid_labels)

In [14]:
df

,target_id,n_matched,mean_rmsd,median_rmsd,skipped
0,8ZNQ,20,12.089670,10.631836,False
1,9CFN,30,23.342223,19.912974,False
2,9E74,169,55.871267,44.760217,False
3,9E75,102,31.775022,26.573829,False
4,9E9Q,98,6.065607,2.974038,False
5,9EBP,55,29.341353,23.072851,False
6,9G4J,240,7.455945,4.934741,False
7,9G4P,43,31.408722,23.501433,False
8,9G4Q,51,40.336481,38.792015,False
9,9G4R,31,26.004572,22.050206,False


In [16]:
print('Validation summary:')
print(summary)

Validation summary:
{'n_targets': 28, 'n_evaluated': 28, 'mean_rmsd': 36.195305345602634, 'median_rmsd': 30.48877698800794}


## Predict on test set

In [18]:
test_seq = load_sequences(os.path.join(data_dir, 'test_sequences.csv'))

In [19]:
# NOTE: 並列化の意味は小さいかも
test_pred = generate_submission(test_seq, repo, seq_identity, n_structures=5, n_jobs=4)
print('submission rows:', len(test_pred))

test_pred.head()

submission rows: 9762


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,12.133,-11.862,1.757,166.708,162.080,161.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8ZNQ_2,C,2,NaN,NaN,NaN,167.160,157.024,159.502,193.176,143.370,155.162,NaN,NaN,NaN,9.981,-13.553,-4.518
2,8ZNQ_3,C,3,NaN,NaN,NaN,164.384,147.392,163.683,188.045,141.202,155.911,NaN,NaN,NaN,8.575,-8.310,5.003
3,8ZNQ_4,G,4,11.260,-7.167,5.547,160.163,145.020,166.617,NaN,NaN,NaN,-15.397,-2.768,4.068,12.275,-5.392,7.117
4,8ZNQ_5,U,5,12.239,-0.828,5.049,155.045,145.136,168.216,183.603,138.622,153.965,-6.594,-8.409,3.503,17.668,-2.604,8.109


## Submission

In [20]:
# Save submission
out_path = os.path.abspath(os.path.join(os.getcwd(),  'submission.csv'))
test_pred.to_csv(out_path, index=False)
print('saved ->', out_path)

saved -> /Users/tatsuki/work/kaggle/kauto/competitions/rna2/experiments/submission.csv
